In [1]:
from torchvision import datasets, transforms
train = datasets.MNIST('data', train=True,  download=True, transform=transforms.ToTensor())
test  = datasets.MNIST('data', train=False, download=True, transform=transforms.ToTensor())

In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader

torch.manual_seed(0)
train_dl = DataLoader(train, batch_size=128, shuffle=True)
test_dl = DataLoader(test, batch_size=1000)

# Network: 784 -> 64 -> 10
model = nn.Sequential(
    nn.Flatten(), 
    nn.Linear(784, 16), nn.ReLU(), 
    nn.Linear(16, 10)
)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

def accuracy():
    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in test_dl:
            correct += (model(x).argmax(1) == y).sum().item()
    return correct / len(test)

for epoch in range(5):
    model.train()
    for x, y in train_dl:
        opt.zero_grad()
        loss_fn(model(x), y).backward()
        opt.step()
    print(f"epoch {epoch + 1}: test accuracy {accuracy():.4f}")

print("params:", sum(p.numel() for p in model.parameters()))

epoch 1: test accuracy 0.9094


epoch 2: test accuracy 0.9222


epoch 3: test accuracy 0.9292


epoch 4: test accuracy 0.9319


epoch 5: test accuracy 0.9349
params: 12730


In [3]:
import numpy as np

# Raw float32 weights, exactly as trained. No quantisation yet: that is a design decision.
# Keys are PyTorch's names for the Sequential: '1.weight', '1.bias', '3.weight', '3.bias'
weights = {k: v.detach().numpy() for k, v in model.state_dict().items()}
np.savez('weights_float.npz', **weights)

for k, v in weights.items():
    print(k, v.shape, v.dtype, f"range [{v.min():.3f}, {v.max():.3f}]")

1.weight (16, 784) float32 range [-0.661, 0.613]
1.bias (16,) float32 range [-0.281, 0.455]
3.weight (10, 16) float32 range [-0.981, 0.659]
3.bias (10,) float32 range [-0.477, 0.302]


In [4]:
# Integer-only inference after quantising the float-trained model: weights to w_bits,
# hidden activations to a_bits (two separate choices).
# From the pixels to the argmax, every operation is integer arithmetic.
import numpy as np

SHIFT = 32  # requantise as (acc * M) >> SHIFT with M an integer: a multiply and a shift, no divide

def imatmul(a, b):
    """Exact integer a @ b: int64 when it provably cannot overflow, Python big ints otherwise."""
    bound = int(np.abs(a).max()) * int(np.abs(b).max()) * a.shape[1]
    if bound < 2 ** 63:
        return a.astype(np.int64) @ b.astype(np.int64)
    return a.astype(object) @ b.astype(object)

def to_int(w, scale, qmax=None):
    """round(w / scale) as exact Python ints, clamped to +-qmax if given."""
    q = [int(v) for v in np.rint(w / scale).flat]
    if qmax is not None:
        q = [max(-qmax, min(qmax, v)) for v in q]
    return np.array(q, dtype=object).reshape(w.shape)

def bits_needed(a):
    return int(np.abs(a).max()).bit_length() + 1  # +1 for the sign bit

sd = {k: v.numpy().astype(np.float64) for k, v in model.state_dict().items()}
X_cal = train.data.numpy().reshape(-1, 784)[:5000].astype(np.int64)   # calibration: train images only
X_test = test.data.numpy().reshape(-1, 784).astype(np.int64)          # raw pixels 0..255, scale 1/255
Y_test = test.targets.numpy()

def int_layer1(x, W1q, b1q):
    return np.maximum(imatmul(x, W1q.T) + b1q, 0)  # accumulate, add bias, ReLU: all integer

def build_int_net(w_bits, a_bits):
    """Quantise weights to `w_bits` and hidden activations to `a_bits`; return every integer the hardware needs."""
    qmax_w = 2 ** (w_bits - 1) - 1            # symmetric: 4 bits -> -7..7
    qmax = 2 ** (a_bits - 1) - 1              # hidden activations clamp to 0..qmax after ReLU
    sw1, sw2 = np.abs(sd['1.weight']).max() / qmax_w, np.abs(sd['3.weight']).max() / qmax_w
    W1q, W2q = to_int(sd['1.weight'], sw1, qmax_w), to_int(sd['3.weight'], sw2, qmax_w)
    s_acc1 = sw1 / 255                        # scale carried by the layer-1 accumulator
    b1q = to_int(sd['1.bias'], s_acc1)        # bias stored in the accumulator's scale
    # Calibrate the rescale: map the largest layer-1 activation seen on train images to qmax.
    max_acc = int(np.abs(int_layer1(X_cal, W1q, b1q)).max())
    M = (qmax << SHIFT) // max_acc            # integer multiplier
    s_h = max_acc * s_acc1 / qmax             # real value of one hidden-activation unit
    b2q = to_int(sd['3.bias'], sw2 * s_h)
    return dict(w_bits=w_bits, a_bits=a_bits, qmax_w=qmax_w, qmax=qmax, W1=W1q, b1=b1q, W2=W2q, b2=b2q, M=M, sw1=sw1, sw2=sw2, s_h=s_h)

def run_int_net(net, x):
    acc1 = int_layer1(x, net['W1'], net['b1'])
    h = np.minimum((acc1.astype(object) * net['M'] + (1 << (SHIFT - 1))) // (1 << SHIFT), net['qmax'])
    acc2 = imatmul(h, net['W2'].T) + net['b2']
    return acc1, h, acc2

def int_inference(w_bits, a_bits):
    net = build_int_net(w_bits, a_bits)
    acc1, h, acc2 = run_int_net(net, X_test)
    pred = np.array([int(np.argmax(r)) for r in acc2])
    assert all(isinstance(v, int) for v in acc2.flat[:50]), "scores must be integers"
    return (pred == Y_test).mean(), bits_needed(acc1), bits_needed(acc2)

print(f"float32 : {accuracy():.4f}")
for w_bits, a_bits in ((4, 8), (4, 4), (8, 8), (16, 16), (32, 32), (64, 64)):
    acc, b1, b2 = int_inference(w_bits, a_bits)
    print(f"w{w_bits:<2} a{a_bits:<2} : {acc:.4f}   accumulator bits observed: layer1 {b1}, layer2 {b2}")

float32 : 0.9349


w4  a8  : 0.9275   accumulator bits observed: layer1 17, layer2 12


w4  a4  : 0.9019   accumulator bits observed: layer1 17, layer2 8


w8  a8  : 0.9343   accumulator bits observed: layer1 22, layer2 16


w16 a16 : 0.9349   accumulator bits observed: layer1 30, layer2 32


w32 a32 : 0.9349   accumulator bits observed: layer1 46, layer2 64


w64 a64 : 0.9349   accumulator bits observed: layer1 78, layer2 128


In [5]:
import json, os

WEIGHT_BITS = 4     # decided (Design Spec v0.1)
ACT_BITS = 8        # hidden-activation width: open decision #1, kept at 8 until decided
BIAS_BITS = 32      # bias word width (the accumulators are wider than the weights)
N_VECTORS = 16      # test images exported with their expected results

net = build_int_net(WEIGHT_BITS, ACT_BITS)
for b in list(net['b1']) + list(net['b2']):
    assert abs(int(b)) < 2 ** (BIAS_BITS - 1), "bias does not fit BIAS_BITS"
os.makedirs('export', exist_ok=True)
for f in os.listdir('export'):     # nothing stale survives a width change
    os.remove(os.path.join('export', f))

def write_hex(path, values, nbits):
    """One two's-complement word per line, for $readmemh."""
    digits = (nbits + 3) // 4
    mask = (1 << nbits) - 1
    with open(path, 'w') as f:
        f.write("\n".join(f"{int(v) & mask:0{digits}X}" for v in values) + "\n")

# Layout: row-major. Word index = neuron * n_inputs + input.
write_hex('export/w1.hex', net['W1'].flatten(), WEIGHT_BITS)   # 16 x 784
write_hex('export/w2.hex', net['W2'].flatten(), WEIGHT_BITS)   # 10 x 16
write_hex('export/b1.hex', net['b1'].flatten(), BIAS_BITS)     # 16
write_hex('export/b2.hex', net['b2'].flatten(), BIAS_BITS)     # 10

# Test vectors: raw 8-bit pixels, plus what the integer model produces for them.
xs = X_test[:N_VECTORS]
acc1, h, acc2 = run_int_net(net, xs)
write_hex('export/test_images.hex', xs.flatten(), 8)
json.dump({
    "labels": [int(v) for v in Y_test[:N_VECTORS]],
    "layer1_acc_after_relu": [[int(v) for v in r] for r in acc1],
    "hidden_activations":    [[int(v) for v in r] for r in h],
    "scores":                [[int(v) for v in r] for r in acc2],
    "predicted":             [int(np.argmax(r)) for r in acc2],
}, open('export/expected.json', 'w'), indent=1)

json.dump({
    "weight_bits": WEIGHT_BITS, "act_bits": ACT_BITS, "bias_bits": BIAS_BITS,
    "qmax_weight": net['qmax_w'], "qmax_act": net['qmax'],
    "shift": SHIFT, "M": int(net['M']),
    "layout": "row-major, index = neuron * n_inputs + input; two's complement",
    "w1_shape": [16, 784], "w2_shape": [10, 16],
    "pixel_scale": "1/255", "weight_scales": [float(net['sw1']), float(net['sw2'])],
    "hidden_unit_scale": float(net['s_h']),
    "rescale": "h = min((relu(acc1) * M + 2**(shift-1)) >> shift, qmax_act)",
}, open('export/meta.json', 'w'), indent=1)

print("M =", net['M'], f"({int(net['M']).bit_length()} bits)")
print("weights", net['W1'].min(), net['W1'].max(), "| biases b1", min(net['b1']), max(net['b1']))
print(sorted(os.listdir('export')))

M = 9065027 (24 bits)
weights -7 6 | biases b1 -757 1228
['b1.hex', 'b2.hex', 'expected.json', 'meta.json', 'test_images.hex', 'w1.hex', 'w2.hex']
